### Qwen3 Highlights

* Qwen3 is the latest generation of large language models in Qwen series, offering a comprehensive suite of dense and mixture-of-experts (MoE) models.

* Uniquely support of seamless switching between thinking mode (for complex logical reasoning, math, and coding) and non-thinking mode (for efficient, general-purpose dialogue) within single model, ensuring optimal performance across various scenarios.

* Support of 100+ languages and dialects with strong capabilities for multilingual instruction following and translation.

In [1]:
### ### import Neccesary Library for Large Language Model
! pip install -U bitsandbytes>=0.46.1, outlines


from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import GenerationConfig
from transformers import  BitsAndBytesConfig

import torch
import warnings
warnings.filterwarnings("ignore")


ERROR: Invalid requirement: 'outlines,langchain-text-splitters': Expected end or semicolon (after name and no valid version specifier)
    outlines,langchain-text-splitters
            ^


In [2]:
# Define the official Qwen 2.5 1.5B Instruct repository path
model_id = "Qwen/Qwen3-0.6B"
# 1. Load the appropriate tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
# 2. Load the instruction-tuned model using AutoModel
vanila_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",          # Automatically chooses between float16/bfloat16 based on your GPU
    device_map="auto"            # Automatically handles device placement (CPU/GPU) shards
)



config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [3]:
#### Refering to the Hugging Face Documentation

def generate_text_function(model_type,message,enable_thinking=False):

  generation_config = GenerationConfig(
    temperature=0.1,
    top_p=0.9,
    top_k=2,
    max_new_tokens=32768,
    do_sample=True # Crucial for Qwen models to respect temperature
  )

  text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    Temperature=0.1,
    enable_thinking=enable_thinking # Switches between thinking and non-thinking modes. Default is True.
  )
  model_inputs = tokenizer([text], return_tensors="pt").to(model_type.device)

  # conduct text completion
  generated_ids = model_type.generate(
      **model_inputs,
      generation_config=generation_config,
  )
  output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

  # parsing thinking content
  try:
      # rindex finding 151668 (</think>)
      index = len(output_ids) - output_ids[::-1].index(151668)
  except ValueError:
      index = 0

  content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")
  if enable_thinking:
    thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
    return content,thinking_content
  else:
    return content,None

In [4]:
#### Output with Enable Thing Mode

messages = [
    {"role": "system", "content": "You are a helpful and concise assistant."},
    {"role": "user", "content": "Give me a short introduction to large language model."}
]
content,thinking_content = generate_text_function(vanila_model,messages,enable_thinking=True)


print("\nOutput:---",content)
print("\nthinking_content:---",thinking_content)




Output:--- A large language model (LLM) is an AI system designed to understand and generate human language. It can process text, answer questions, and create content, making it versatile for tasks like writing, translation, and content creation.

thinking_content:--- <think>
Okay, the user wants a short introduction to a large language model. Let me start by recalling what I know. Large language models are AI systems designed to understand and generate human language. They can process text, answer questions, and even create content. I should mention their key features like understanding context, generating text, and being able to handle various tasks. Also, it's important to highlight their applications in different fields. Let me make sure the introduction is concise and covers the main points without getting too technical. Avoid jargon, keep it simple. Check for any missing elements and ensure clarity.
</think>


In [5]:
#### Output with Enable Thing Mode

messages = [
    {"role": "system", "content": "You are a helpful and concise assistant."},
    {"role": "user", "content": "Give me a short introduction to large language model."}
]
content,thinking_content = generate_text_function(vanila_model,messages,enable_thinking=False)


print("\nOutput:---",content)
print("\nthinking_content:---",thinking_content)




Output:--- A large language model is a type of artificial intelligence designed to understand and generate human language. It can process text, comprehend context, and produce coherent responses.

thinking_content:--- None


### Load Model With Quantization
* Quantization is a technique to reduce the computational and memory costs of running inference by representing the weights and activations with low-precision data types like 8-bit integer (int8) instead of the usual 32-bit floating point (float32).


* educing the number of bits means the resulting model requires less memory storage, consumes less energy (in theory), and operations like matrix multiplication can be performed much faster with integer arithmetic. It also allows to run models on embedded devices, which sometimes only support integer data types.

In [6]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,                  # Enables 4-bit quantization
    bnb_4bit_quant_type="nf4",          # Uses Normalized Float 4 (best for LLMs)
    bnb_4bit_compute_dtype=torch.bfloat16, # Sets internal computation precision
    bnb_4bit_use_double_quant=True      # Quantizes the quantization constants for extra VRAM savings
)

# 2. Load the instruction-tuned model using AutoModel
model_quant = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",          # Automatically chooses between float16/bfloat16 based on your GPU
    device_map="auto",
    quantization_config=quantization_config
)

print("\n\nModel Confugurtion Before Quantization")
bytes_used = vanila_model.get_memory_footprint()
print(f"Model VRAM Usage: {bytes_used / (1024**3):.2f} GB")

print("\nModel Confugurtion After Quantization")
bytes_used = model_quant.get_memory_footprint()
print(f"Model VRAM Usage: {bytes_used / (1024**3):.2f} GB")


ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful and concise assistant."},
    {"role": "user", "content": "Give me a short introduction to large language model."}
]
content,thinking_content = generate_text_function(model_quant,messages,enable_thinking=True)




print("Output:---",content)
print("\nthinking_content:---",thinking_content)



### Custom system promt and user promt for Named Entity Recognisation

In [ ]:
system_promt = '''You are an expert Named Entity Recognition (NER) and Information Extraction system.

Your task is to extract the following entities from the provided text:

1. PERSON_NAME
2. COMPANY_NAME
3. DATE_OF_BIRTH
4. DATE_OF_JOINING

Instructions:

* Extract only information explicitly present in the text.
* Do not infer or hallucinate missing values.
* If an entity is not found, return null.
* Return the output strictly in JSON format.
* If multiple values are found for an entity, return them as a list.

Output Format:

{
"PERSON_NAME": [],
"COMPANY_NAME": [],
"DATE_OF_BIRTH": [],
"DATE_OF_JOINING": []
}
'''

user_prompt = '''Extract the following entities from the provided text:

* PERSON_NAME
* COMPANY_NAME
* DATE_OF_BIRTH
* DATE_OF_JOINING

Text:
{document_text}

Return the result in the specified JSON format only.
'''

input_text = 'John Smith joined Infosys on 15 March 2022. His date of birth is 12 July 1990.'

user_prompt = user_prompt.format(document_text = input_text)

messages = [
    {"role": "system", "content":system_promt},
    {"role": "user", "content": user_prompt}
]
content,thinking_content = generate_text_function(model_quant,messages,enable_thinking=True)




print("Output:---",content)
print("\nthinking_content:---",thinking_content)

### Structured Output With Pydantic Class with Hugging Face outline

In [ ]:
from pydantic import BaseModel,Field
import outlines


class EmployeeClass(BaseModel):
    Employee: str  = Field(description="The full legal name of the Employee.")
    Campany_Name: str = Field(description="The Campany name of the Employee.")
    DATE_OF_BIRTH: str = Field(description="The Date of Brth of the Employee.")
    JOINING_DATE: str = Field(description="The Joining Date of the Employee.")


In [ ]:
def generate_structure_output(text):


  outlines_model = outlines.from_transformers(
    vanila_model,
    tokenizer
  )
  generator = outlines.Generator(
    outlines_model,
    EmployeeClass
  )
  result = generator(
    f"Extract employee information:\n{text}",max_new_tokens=32768)

  return result

structure_output = generate_structure_output(input_text)

print(structure_output)

{"Employee": "John Smith", "Campany_Name": "Infosys", "DATE_OF_BIRTH": "1990-07-12", "JOINING_DATE": "2022-03-15"}


### Structured Output With Pydantic Class with Hugging Face outline Enable Thinking Mode

In [ ]:
def generate_structure_enable_think_output(messages):
  prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True
  )
  outlines_model = outlines.from_transformers(
    model_quant,
    tokenizer
  )
  generator = outlines.Generator(
      outlines_model,
      EmployeeClass,
  )

  result = generator(prompt,max_new_tokens=32768)

  employee = EmployeeClass.model_validate_json(result)
  return employee


messages = [
    {"role": "system", "content":system_promt},
    {"role": "user", "content": user_prompt}
]
structure_output = generate_structure_enable_think_output(messages)

print(structure_output)

Employee='John Smith' Campany_Name='Infosys' DATE_OF_BIRTH='12 July 1990' JOINING_DATE='15 March 2022'
